# Prueba exploratoria de geocodificación con geopy

Este notebook evalúa si el servicio Nominatim, consultado mediante `geopy`, permite recuperar coordenadas para una selección de prefijos postales de clientes que no encuentran correspondencia en la tabla de geolocalización de Olist.

La prueba es exploratoria y no modifica el criterio metodológico del EDA principal. Se distinguen dos resultados:

1. **Consulta postal:** utiliza prefijo postal, ciudad, estado y país.
2. **Consulta territorial:** utiliza únicamente ciudad, estado y país como referencia comparativa.

Una coordenada obtenida mediante la segunda estrategia representa una ubicación territorial aproximada y no debe interpretarse como una coordenada del prefijo postal.

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

# Busca la carpeta del proyecto si el kernel se inicia desde Repos, TFI_ITBA o notebooks/.
candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd() / "TFI_ITBA"]
PROYECTO = next((ruta for ruta in candidatos if (ruta / "data" / "parquet").exists()), None)
if PROYECTO is None:
    raise FileNotFoundError("No se encontró la carpeta data/parquet del proyecto.")

DATA_DIR = PROYECTO / "data" / "parquet"
print(f"Carpeta de datos: {DATA_DIR.resolve()}")

Carpeta de datos: C:\Users\daniel.fernandez\Desktop\Repos\TFI_ITBA\data\parquet


## 1. Identificación de prefijos sin correspondencia

Se reproducen los controles del EDA principal para identificar los prefijos presentes en clientes pero ausentes en la tabla de geolocalización.

In [2]:
clientes = pd.read_parquet(DATA_DIR / "olist_customers_dataset.parquet")
geolocalizacion = pd.read_parquet(DATA_DIR / "olist_geolocation_dataset.parquet")

prefijos_con_geo = set(geolocalizacion["geolocation_zip_code_prefix"].unique())
clientes_sin_geo = clientes.loc[
    ~clientes["customer_zip_code_prefix"].isin(prefijos_con_geo)
].copy()
clientes_sin_geo["zip_prefijo"] = (
    clientes_sin_geo["customer_zip_code_prefix"].astype("string").str.zfill(5)
)

casos = (
    clientes_sin_geo
    .groupby(["zip_prefijo", "customer_city", "customer_state"], as_index=False)
    .size()
    .rename(columns={"size": "cantidad_clientes"})
    .sort_values("cantidad_clientes", ascending=False)
)

print(f"Registros de clientes sin geolocalización: {len(clientes_sin_geo):,}")
print(f"Prefijos diferentes sin geolocalización: {casos['zip_prefijo'].nunique():,}")
display(casos.head(10))

Registros de clientes sin geolocalización: 278
Prefijos diferentes sin geolocalización: 157


,zip_prefijo,customer_city,customer_state,cantidad_clientes
69,70686,brasilia,DF,15
95,72005,brasilia,DF,13
86,71919,brasilia,DF,10
130,73255,brasilia,DF,7
104,72300,brasilia,DF,6
133,73369,brasilia,DF,5
84,71884,brasilia,DF,5
94,72002,brasilia,DF,5
135,73401,brasilia,DF,5
81,71676,brasilia,DF,4


## 2. Selección de casos para la prueba

La consulta se limita a cinco casos y selecciona, cuando es posible, un prefijo de cada estado. Esta restricción evita realizar una geocodificación masiva y permite observar si el comportamiento cambia entre territorios.

In [3]:
N_CASOS = 5

casos_prueba = (
    casos
    .groupby("customer_state", sort=False, as_index=False)
    .head(1)
    .head(N_CASOS)
    .reset_index(drop=True)
)

display(casos_prueba)

,zip_prefijo,customer_city,customer_state,cantidad_clientes
0,70686,brasilia,DF,15
1,41098,salvador,BA,3
2,29949,sao mateus,ES,3
3,72821,luziania,GO,3
4,65137,maioba,MA,3


## 3. Configuración del geocodificador

`geopy` funciona como cliente de servicios externos. En esta prueba se utiliza el servidor público de Nominatim y se establece un intervalo mínimo de 1,1 segundos entre consultas. El identificador de aplicación permite que el servicio reconozca el origen de las solicitudes.

La política vigente de Nominatim establece un máximo absoluto de una solicitud por segundo, desalienta los procesos masivos y exige identificar la aplicación. Por ese motivo, este notebook está diseñado para una verificación acotada y manual. Los resultados se conservan en `outputs/geopy_prueba_cache.csv` para evitar repetir consultas ya realizadas.

In [4]:
geolocator = Nominatim(
    user_agent="tfi_itba_eda_geopy_daniel_fernandez",
    timeout=15,
)

geocodificar = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,
    max_retries=1,
    error_wait_seconds=2,
    swallow_exceptions=True,
)

print("Geocodificador configurado para una prueba acotada.")

Geocodificador configurado para una prueba acotada.


## 4. Consulta postal y consulta territorial

Para cada caso se realiza primero una consulta estructurada con el prefijo postal. La coincidencia se considera postal únicamente si la respuesta contiene un código postal cuyo comienzo coincide con los cinco dígitos consultados.

También se consulta la combinación ciudad-estado para comprobar si el servicio puede devolver una referencia territorial aun cuando no identifique el prefijo. Las coordenadas de ambas estrategias se guardan en columnas separadas.

In [5]:
def solo_digitos(valor):
    if valor is None:
        return None
    return re.sub(r"\D", "", str(valor))


def datos_ubicacion(ubicacion):
    if ubicacion is None:
        return {
            "encontrada": False,
            "latitud": None,
            "longitud": None,
            "codigo_postal_respuesta": None,
            "descripcion": None,
        }

    direccion = ubicacion.raw.get("address", {})
    return {
        "encontrada": True,
        "latitud": float(ubicacion.latitude),
        "longitud": float(ubicacion.longitude),
        "codigo_postal_respuesta": direccion.get("postcode"),
        "descripcion": ubicacion.address,
    }


CACHE_PATH = PROYECTO / "outputs" / "geopy_prueba_cache.csv"
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

if CACHE_PATH.exists():
    cache = pd.read_csv(CACHE_PATH, dtype={"zip_prefijo": "string"})
else:
    cache = pd.DataFrame()

resultados = []

for caso in casos_prueba.itertuples(index=False):
    if not cache.empty:
        fila_cache = cache.loc[
            cache["zip_prefijo"].eq(caso.zip_prefijo)
            & cache["ciudad_olist"].eq(caso.customer_city)
            & cache["estado_olist"].eq(caso.customer_state)
        ]
        if not fila_cache.empty:
            resultados.append(fila_cache.iloc[0].to_dict())
            continue

    consulta_postal = {
        "postalcode": caso.zip_prefijo,
        "city": caso.customer_city,
        "state": caso.customer_state,
        "country": "Brazil",
    }
    ubicacion_postal = geocodificar(
        consulta_postal,
        exactly_one=True,
        addressdetails=True,
        country_codes="br",
        language="pt-BR",
    )
    postal = datos_ubicacion(ubicacion_postal)

    codigo_respuesta = solo_digitos(postal["codigo_postal_respuesta"])
    coincide_prefijo = bool(
        codigo_respuesta and codigo_respuesta.startswith(caso.zip_prefijo)
    )

    consulta_territorial = f"{caso.customer_city}, {caso.customer_state}, Brasil"
    ubicacion_territorial = geocodificar(
        consulta_territorial,
        exactly_one=True,
        addressdetails=True,
        country_codes="br",
        language="pt-BR",
    )
    territorial = datos_ubicacion(ubicacion_territorial)

    resultados.append(
        {
            "zip_prefijo": caso.zip_prefijo,
            "ciudad_olist": caso.customer_city,
            "estado_olist": caso.customer_state,
            "cantidad_clientes": caso.cantidad_clientes,
            "consulta_postal_encontrada": postal["encontrada"],
            "codigo_postal_respuesta": postal["codigo_postal_respuesta"],
            "coincide_prefijo": coincide_prefijo,
            "latitud_postal": postal["latitud"],
            "longitud_postal": postal["longitud"],
            "descripcion_postal": postal["descripcion"],
            "consulta_territorial_encontrada": territorial["encontrada"],
            "latitud_territorial": territorial["latitud"],
            "longitud_territorial": territorial["longitud"],
            "descripcion_territorial": territorial["descripcion"],
        }
    )

resultados_geopy = pd.DataFrame(resultados)
resultados_geopy["misma_coordenada_que_ciudad_estado"] = (
    resultados_geopy["latitud_postal"].round(6).eq(
        resultados_geopy["latitud_territorial"].round(6)
    )
    & resultados_geopy["longitud_postal"].round(6).eq(
        resultados_geopy["longitud_territorial"].round(6)
    )
)

cache_actualizado = pd.concat([cache, resultados_geopy], ignore_index=True)
cache_actualizado = cache_actualizado.drop_duplicates(
    subset=["zip_prefijo", "ciudad_olist", "estado_olist"],
    keep="last",
)
cache_actualizado.to_csv(CACHE_PATH, index=False)
print(f"Resultados conservados en: {CACHE_PATH.resolve()}")
display(resultados_geopy)

Resultados conservados en: C:\Users\daniel.fernandez\Desktop\Repos\TFI_ITBA\outputs\geopy_prueba_cache.csv


,zip_prefijo,ciudad_olist,estado_olist,cantidad_clientes,consulta_postal_encontrada,codigo_postal_respuesta,coincide_prefijo,latitud_postal,longitud_postal,descripcion_postal,consulta_territorial_encontrada,latitud_territorial,longitud_territorial,descripcion_territorial,misma_coordenada_que_ciudad_estado
0,70686,brasilia,DF,15,True,70391-900,False,-15.793987,-47.882800,"Brasília, Plano Piloto, Distrito Federal, Regi...",True,-15.793987,-47.882800,"Brasília, Plano Piloto, Distrito Federal, Regi...",True
1,41098,salvador,BA,3,True,NaN,False,-12.982250,-38.481277,"Salvador, Bahia, Região Nordeste, Brasil",True,-12.982250,-38.481277,"Salvador, Bahia, Região Nordeste, Brasil",True
2,29949,sao mateus,ES,3,True,NaN,False,-18.796504,-39.921237,"São Mateus, Espírito Santo, Região Sudeste, Br...",True,-18.796504,-39.921237,"São Mateus, Espírito Santo, Região Sudeste, Br...",True
3,72821,luziania,GO,3,True,NaN,False,-16.253546,-47.950028,"Luziânia, Goiás, Região Centro-Oeste, Brasil",True,-16.253546,-47.950028,"Luziânia, Goiás, Região Centro-Oeste, Brasil",True
4,65137,maioba,MA,3,True,65137-000,True,-2.524325,-44.178247,"Maioba, Paço do Lumiar, Maranhão, Região Norde...",True,-2.524325,-44.178247,"Maioba, Paço do Lumiar, Maranhão, Região Norde...",True


## 5. Síntesis de la prueba

El cuadro siguiente separa las respuestas obtenidas de las coincidencias postales verificables. Una respuesta del geocodificador no constituye por sí sola evidencia de que la coordenada corresponda al prefijo consultado.

In [6]:
resumen = pd.Series(
    {
        "casos_consultados": len(resultados_geopy),
        "respuestas_consulta_postal": int(resultados_geopy["consulta_postal_encontrada"].sum()),
        "coincidencias_postales_verificadas": int(resultados_geopy["coincide_prefijo"].sum()),
        "respuestas_ciudad_estado": int(resultados_geopy["consulta_territorial_encontrada"].sum()),
        "consultas_postales_iguales_al_centroide_territorial": int(
            resultados_geopy["misma_coordenada_que_ciudad_estado"].sum()
        ),
    },
    name="cantidad",
).to_frame()
display(resumen)

if resultados_geopy["coincide_prefijo"].all():
    print("Todos los casos de prueba produjeron una coincidencia postal verificable.")
elif resultados_geopy["coincide_prefijo"].any():
    print("La recuperación postal fue parcial y requiere validación caso por caso.")
else:
    print("No se obtuvieron coincidencias postales verificables en los casos seleccionados.")

if resultados_geopy["consulta_territorial_encontrada"].any():
    print("Las respuestas por ciudad y estado sólo representan referencias territoriales aproximadas.")

,cantidad
casos_consultados,5
respuestas_consulta_postal,5
coincidencias_postales_verificadas,1
respuestas_ciudad_estado,5
consultas_postales_iguales_al_centroide_territorial,5


La recuperación postal fue parcial y requiere validación caso por caso.
Las respuestas por ciudad y estado sólo representan referencias territoriales aproximadas.


## 6. Alcance de los resultados

Esta prueba permite evaluar la factibilidad técnica de consultar un geocodificador externo, pero no valida automáticamente una fuente para el dataset analítico. Antes de incorporar coordenadas externas sería necesario analizar la cobertura sobre todos los casos faltantes, la consistencia con las coordenadas de Olist, la reproducibilidad del servicio, sus condiciones de uso y la diferencia de escala entre un prefijo postal y un centroide municipal.